In [1]:
import pandas as pd
import os
import re

In [2]:
def combine_snapshots(pokemon_name, folder="../pokemon_snapshots"):
    dfs = []

    # Normalize name for matching
    safe_name = pokemon_name.replace(" ", "_").lower()

    for filename in os.listdir(folder):
        if filename.endswith(".csv") and filename.startswith(safe_name):
            filepath = os.path.join(folder, filename)
            df = pd.read_csv(filepath)

            # Extract date from filename: psyduck_pricing_2026-07-10.csv
            match = re.search(r"(\d{4}-\d{2}-\d{2})", filename)
            snapshot_date = match.group(1) if match else None
            df["snapshot_date"] = snapshot_date

            dfs.append(df)

    if not dfs:
        raise ValueError(f"No CSV snapshots found for Pokémon: {pokemon_name}")

    combined = pd.concat(dfs, ignore_index=True)

    # Sort by card + date
    combined = combined.sort_values(["id", "snapshot_date"]).reset_index(drop=True)

    # Remove exact duplicates
    combined = combined.drop_duplicates()

    return combined

In [ ]:
## Option for excluding specific sets ##

# EXCLUDED_SETS = {
#     "Promos-A",
#     "Genetic Apex",
#     "Mythical Island",
#     "Space-Time Smackdown",
#     "Triumphant Light",
#     "Shining Revelry",
#     "Celestial Guardians",
#     "Extradimensional Crisis",
#     "Eevee Grove",
#     "Wisdom of Sea and Sky",
#     "Secluded Springs",
#     "Deluxe Pack: ex",

#     "Promos-B",
#     "Mega Rising",
#     "Crimson Blaze",
#     "Fantastical Parade",
#     "Paldean Wonders",
#     "Mega Shine",
#     "Pulsing Aura",
#     "Paradox Drive",
#     "Everyday Wonders",
#     "Ruler of the Skies",
# }

# def remove_tcg_pocket_cards(df):
#     if "set" not in df.columns:
#         return df  # nothing to filter

#     return df[~df["set"].isin(EXCLUDED_SETS)].reset_index(drop=True)

In [ ]:

def preprocess_combined_df(df, pokemon_name, output_folder="../data"):
    # Removing columns that are entirely NaN
    df = df.dropna(axis=1, how="all")

    # Removing columns that are entirely empty strings
    empty_cols = [
        col for col in df.columns
        if df[col].astype(str).str.strip().eq("").all()
    ]
    df = df.drop(columns=empty_cols)

    # Identify pricing columns
    pricing_cols = [col for col in df.columns if col.startswith(("cm_", "tp_"))]

    # Removing rows with no pricing at all
    df = df.dropna(subset=pricing_cols, how="all").reset_index(drop=True)

    # --- SAVE CLEANED DATAFRAME ---
    os.makedirs(output_folder, exist_ok=True)

    safe_name = pokemon_name.replace(" ", "_").lower()
    output_path = os.path.join(output_folder, f"{safe_name}_combined_clean.csv")

    df.to_csv(output_path, index=False)

    return df, output_path


In [ ]:
# Loop through Pokemon snapshots
pokemon_list = ["psyduck", "pikachu", "kadabra"]

for pokemon in pokemon_list:
    combined_df = combine_snapshots(pokemon)
    clean_df, output_path = preprocess_combined_df(combined_df, pokemon)
    print(f"Saved cleaned combined file for {pokemon} to: {output_path}\n")


Saved cleaned combined file for golduck to: ../data/golduck_combined_clean.csv



In [ ]:
# loop through set snapshots
set_list = ["chaos_rising", "pitch_black"]

for set_name in set_list:
    combined_set = combine_snapshots(set_name, folder="../set_snapshots")
    clean_set, output_path = preprocess_combined_df(combined_set, set_name)
    print(f"Saved cleaned combined file for {set_name} to: {output_path}\n")
